## 4. 两个入口 × 两种输入模式

> 来源：[Streaming Input](https://code.claude.com/docs/en/agent-sdk/streaming-vs-single-mode)、[Agent SDK reference - Python](https://code.claude.com/docs/en/agent-sdk/python)


### 4.1 两个入口：`query()` 与 `ClaudeSDKClient`

| 维度 | `query()` | `ClaudeSDKClient` |
|---|---|---|
| 会话 | 每次调用**新开**一个 session | 复用同一 session |
| 对话 | 单次任务 | 同一上下文里连续追问 |
| 流式输入 | ✅（prompt 传 async generator） | ✅ |
| **打断 (interrupt)** | ❌ | ✅ |
| 续接上下文 | 手动（传 `resume` / `continue_conversation`） | 自动 |
| 适用 | 一次性任务、无状态环境（如 lambda） | 持续交互 / 需要中途打断 |

一句话：**能一句话说完的任务用 `query()`，要来回聊或要中途叫停用 `ClaudeSDKClient`**。


### 4.2 两种输入模式：单条消息 vs 流式输入

`prompt` 参数接受两种输入，对应官方说的两种输入模式。两种就是传参形式的差别：


In [ ]:
# 单条消息（single message input）：prompt 传字符串
async for message in query(prompt="总结这个项目", options=options):
    ...


# 流式输入（streaming input mode，官方推荐）：prompt 传 async generator，
# agent 变成长生命周期进程，可以边跑边喂新消息
async def message_stream():
    yield {"type": "user", "message": {"role": "user", "content": "分析这份日志"}}
    yield {"type": "user", "message": {"role": "user", "content": "重点看错误行"}}


async for message in query(prompt=message_stream(), options=options):
    ...


- **单条消息**：简单，但官方明确列出四个不支持：消息内附图、动态消息排队、实时打断、自然多轮对话。多轮只能靠 `continue_conversation=True` 或 `resume` 反复拉起。
- **流式输入**：支持附图、排队、打断、权限回调。`ClaudeSDKClient` 内部就是这种模式。消息 dict 的完整格式与可运行示例见 §16「流式输入」。

两者的共同点比差异更重要：**输入的是"任务"不是"问题"，输出的是"异步消息流"不是"一段字符串"**。这决定了 §7「怎么读消息流」是绕不过去的核心技能。

> [!warning] 失败时先给 ResultMessage，随后仍会抛异常
> 运行失败（如触发 `max_turns`）时，`query()` 会先正常 yield 出最后一条 `ResultMessage`（`subtype='error_max_turns'`），**之后仍会从 `async for` 处抛出异常**。异常会跳过循环后面的代码，所以循环要包 `try/except`；异常抛出前收到的消息照常可用。下方 cell 用 `max_turns=1` 强行触限可复现。注意这个"抛异常"是单次 `query()` 的行为（底层 Claude Code 进程也随之以非零码退出）；**流式输入 session 在错误 result 之后仍然存活**，可以继续往里发消息（§16「流式输入」）。

In [2]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage


async def demo_error_semantics():
    """用 max_turns=1 强行触限，复现"先 yield ResultMessage、后 raise"的顺序。"""
    try:
        async for message in query(
            prompt="List the files here, then read and summarize each one.",  # 1 turn 做不完
            options=ClaudeAgentOptions(
                cwd=".", allowed_tools=["Bash", "Read"], max_turns=1
            ),
        ):
            if isinstance(message, ResultMessage):
                print(message)
                # 这行先执行：subtype='error_max_turns'，result 不可用
                print("[result]", message.subtype, "is_error=", message.is_error)
    except Exception as e:
        # 然后才走到这里——不包 try/except 的话，循环后面的代码不会执行
        print("[raised after result]", type(e).__name__, e)


await demo_error_semantics()

ResultMessage(subtype='error_max_turns', duration_ms=5492, duration_api_ms=4500, is_error=True, num_turns=2, session_id='ff9b0f64-387e-4ab8-9f70-f8895f2a5752', stop_reason='tool_use', total_cost_usd=0.10272599999999998, usage={'input_tokens': 2891, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 67116, 'output_tokens': 134, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'inference_geo': '', 'iterations': [], 'speed': 'standard'}, result=None, structured_output=None, model_usage={'claude-fable-5': {'inputTokens': 2891, 'outputTokens': 134, 'cacheReadInputTokens': 67116, 'cacheCreationInputTokens': 0, 'webSearchRequests': 0, 'costUSD': 0.10272599999999998, 'contextWindow': 200000, 'maxOutputTokens': 64000}}, permission_denials=[], deferred_tool_use=None, errors=['Reached maximum number of turns (1)'], api_error_status=None, uuid='29d7f43c-4